# Go Callgraph Dependency Analysis
Reads results produced by the batch analysis script and consolidates them into charts and tables for the thesis.

**Directory layout expected:**
```
Results/
  <project-name>/
    shallow/            (depth=1, stdlib included)
    deep/               (depth=-1, stdlib included)
    shallow_no_stdlib/  (depth=1, stdlib excluded)
    deep_no_stdlib/     (depth=-1, stdlib excluded)
      report.json
```

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Aesthetics ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
CASE_ORDER   = ["shallow", "shallow_no_stdlib", "deep", "deep_no_stdlib"]
CASE_LABELS  = {
    "shallow"          : "Depth 1",
    "shallow_no_stdlib": "Depth 1 (no stdlib)",
    "deep"             : "Full depth",
    "deep_no_stdlib"   : "Full depth (no stdlib)",
}
PALETTE = sns.color_palette("muted", 4)
CASE_COLORS = dict(zip(CASE_ORDER, PALETTE))

RESULTS_DIR = Path.home() / "UvA" / "Thesis" / "Results"
print(f"Results directory: {RESULTS_DIR}")
print(f"Exists: {RESULTS_DIR.exists()}")

## 1. Load all results

In [ ]:
def find_report(case_dir: Path) -> Path | None:
    """Find the JSON report inside a case directory.
    The tool writes a single JSON file; we accept any name."""
    for p in case_dir.rglob("*.json"):
        return p
    return None


def load_results(results_dir: Path) -> dict:
    """
    Returns a nested dict:
        data[project_name][case_name] = parsed JSON dict
    """
    data = {}
    for project_dir in sorted(results_dir.iterdir()):
        if not project_dir.is_dir():
            continue
        project_name = project_dir.name
        data[project_name] = {}
        for case_name in CASE_ORDER:
            case_dir = project_dir / case_name
            if not case_dir.exists():
                continue
            report_path = find_report(case_dir)
            if report_path is None:
                print(f"  [WARN] No JSON found in {case_dir}")
                continue
            with open(report_path, encoding="utf-8") as f:
                data[project_name][case_name] = json.load(f)
    return data


RAW = load_results(RESULTS_DIR)
print(f"Loaded {len(RAW)} projects:")
for proj, cases in RAW.items():
    print(f"  {proj}: {list(cases.keys())}")

## 2. Package grouping helpers

Go stdlib and third-party packages often share a common top-level prefix  
(e.g. `crypto/rand`, `crypto/rsa` → `crypto`; `github.com/foo/bar/pkg` → `github.com/foo/bar`).  
We offer two granularities:
- **flat** – use the raw package path as-is  
- **grouped** – collapse to a logical dependency name

In [ ]:
# Known single-word stdlib roots that should NOT be split further.
_STDLIB_SINGLE = {
    "bufio","bytes","context","errors","fmt","io","log","math",
    "net","os","path","reflect","regexp","runtime","sort","strconv",
    "strings","sync","syscall","testing","time","unicode","unsafe",
    "compress","container","crypto","database","debug","encoding",
    "expvar","flag","go","hash","html","image","index","internal",
    "mime","plugin","text","unicode",
}

def logical_dep(pkg_path: str) -> str:
    """
    Map a package import path to a logical dependency name.

    Rules (in order):
    1. github.com/owner/repo/...  →  github.com/owner/repo
    2. golang.org/x/name/...     →  golang.org/x/name
    3. gopkg.in/name.v2/...      →  gopkg.in/name.v2
    4. any other domain/.../...  →  first three segments
    5. stdlib (no dot in first segment):
         - multi-part (crypto/rand) → first segment (crypto)
         - single-word             → as-is
    """
    parts = pkg_path.split("/")

    # Third-party: first part contains a dot  →  it's a domain
    if "." in parts[0]:
        if parts[0] in ("github.com", "gitlab.com", "bitbucket.org") and len(parts) >= 3:
            return "/".join(parts[:3])
        if parts[0] in ("golang.org", "google.golang.org") and len(parts) >= 3:
            return "/".join(parts[:3])
        if parts[0] == "gopkg.in" and len(parts) >= 2:
            return "/".join(parts[:2])
        # Generic: keep up to 3 segments
        return "/".join(parts[:3])

    # stdlib
    if len(parts) == 1:
        return pkg_path          # e.g. "fmt"
    return parts[0]              # e.g. "crypto/rand" → "crypto"


# Quick sanity check
tests = [
    "crypto/rand", "crypto/rsa", "net/http", "fmt",
    "github.com/spf13/cobra", "github.com/spf13/cobra/cmd",
    "golang.org/x/tools/go/ssa",
    "gopkg.in/yaml.v3",
]
for t in tests:
    print(f"{t:45s} → {logical_dep(t)}")

## 3. Build a flat DataFrame of per-package metrics

In [ ]:
def compute_package_rows(raw: dict) -> pd.DataFrame:
    """
    One row per (project, case, package).
    Columns: project, case, pkg_path, dep_group, depth,
             func_total, func_unused, func_reached,
             acr, edge_total, is_stdlib
    """
    rows = []
    for project, cases in raw.items():
        for case, report in cases.items():
            reachable_set = set()   # we rebuild from unusedFunctions vs total
            total_funcs   = report.get("totalFunctions", 0)
            reach_funcs   = report.get("reachableFunctions", 0)

            for pkg_path, pkg in report.get("packages", {}).items():
                func_total  = pkg.get("functionCount", 0)
                unused      = pkg.get("unusedFunctions", [])
                func_unused = len(unused)
                func_reached = max(func_total - func_unused, 0)
                acr = func_reached / func_total if func_total > 0 else None

                edge_total = pkg.get("edges", {}).get("total", 0)
                depth      = pkg.get("depth", -1)
                dep_group  = logical_dep(pkg_path)
                is_stdlib  = "." not in pkg_path.split("/")[0]

                rows.append(dict(
                    project      = project,
                    case         = case,
                    pkg_path     = pkg_path,
                    dep_group    = dep_group,
                    depth        = depth,
                    func_total   = func_total,
                    func_unused  = func_unused,
                    func_reached = func_reached,
                    acr          = acr,
                    edge_total   = edge_total,
                    is_stdlib    = is_stdlib,
                ))
    return pd.DataFrame(rows)


PKG_DF = compute_package_rows(RAW)
print(f"Package rows: {len(PKG_DF):,}")
PKG_DF.head()

## 4. Build a flat DataFrame of project-level metrics (RQ1, RQ3)

In [ ]:
def compute_project_rows(raw: dict, pkg_df: pd.DataFrame) -> pd.DataFrame:
    """
    One row per (project, case).
    Columns: project, case, total_funcs, reachable_funcs, reachability_ratio,
             sci (Software Composition Index),
             static_sites, interface_sites, funcvar_sites,
             total_sites, indirect_ratio
    """
    rows = []
    for project, cases in raw.items():
        for case, report in cases.items():
            total   = report.get("totalFunctions", 0)
            reached = report.get("reachableFunctions", 0)

            # SCI: fraction of reachable functions from third-party packages
            subset = pkg_df[
                (pkg_df.project == project) &
                (pkg_df.case    == case)    &
                (~pkg_df.is_stdlib)
            ]
            ext_reached = subset["func_reached"].sum()
            sci = ext_reached / reached if reached > 0 else None

            indirect = report.get("indirect", {})
            static_s = indirect.get("staticCallSites", 0)
            iface_s  = indirect.get("interfaceCallSites", 0)
            fvar_s   = indirect.get("funcVarCallSites", 0)
            total_s  = static_s + iface_s + fvar_s
            indirect_ratio = (iface_s + fvar_s) / total_s if total_s > 0 else None

            rows.append(dict(
                project           = project,
                case              = case,
                total_funcs       = total,
                reachable_funcs   = reached,
                reachability_ratio= reached / total if total > 0 else None,
                sci               = sci,
                static_sites      = static_s,
                interface_sites   = iface_s,
                funcvar_sites     = fvar_s,
                total_sites       = total_s,
                indirect_ratio    = indirect_ratio,
            ))
    return pd.DataFrame(rows)


PROJ_DF = compute_project_rows(RAW, PKG_DF)
print(f"Project rows: {len(PROJ_DF)}")
PROJ_DF

## 5. Utilisation Index (UI) per dependency group  (RQ2)

In [ ]:
def compute_ui(pkg_df: pd.DataFrame) -> pd.DataFrame:
    """
    UI(dep_group) = |R(dep_group)| / |R_total|
    Aggregated first by (project, case, dep_group), then UI computed.
    """
    grp = (
        pkg_df
        .groupby(["project", "case", "dep_group"], as_index=False)
        .agg(
            func_reached = ("func_reached", "sum"),
            func_total   = ("func_total",   "sum"),
            is_stdlib    = ("is_stdlib",     "first"),
        )
    )
    # Total reachable per (project, case)
    total_reached = (
        pkg_df
        .groupby(["project", "case"])["func_reached"]
        .sum()
        .rename("total_reached")
        .reset_index()
    )
    grp = grp.merge(total_reached, on=["project", "case"])
    grp["ui"]  = grp["func_reached"] / grp["total_reached"]
    grp["acr"] = grp["func_reached"] / grp["func_total"].replace(0, np.nan)
    return grp


UI_DF = compute_ui(PKG_DF)
print(f"UI rows: {len(UI_DF)}")
UI_DF.sort_values("ui", ascending=False).head(10)

---
## 6. RQ1 — Reachability ratio per project × configuration

In [ ]:
def plot_reachability_grouped(proj_df: pd.DataFrame):
    cases    = CASE_ORDER
    projects = sorted(proj_df["project"].unique())
    n_proj   = len(projects)
    n_cases  = len(cases)

    x      = np.arange(n_proj)
    width  = 0.8 / n_cases
    offset = np.linspace(-(n_cases-1)/2, (n_cases-1)/2, n_cases) * width

    fig, ax = plt.subplots(figsize=(max(10, n_proj * 1.6), 5))

    for i, case in enumerate(cases):
        sub = proj_df[proj_df["case"] == case].set_index("project")
        vals = [sub.loc[p, "reachability_ratio"] if p in sub.index else np.nan
                for p in projects]
        ax.bar(
            x + offset[i], vals, width,
            label  = CASE_LABELS[case],
            color  = CASE_COLORS[case],
            alpha  = 0.88,
            zorder = 3,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(projects, rotation=30, ha="right")
    ax.set_ylabel("Reachability ratio  (reached / total functions)")
    ax.set_title("RQ1 — Reachability ratio per project and analysis configuration")
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(loc="upper right", framealpha=0.9)
    ax.grid(axis="y", zorder=0)
    plt.tight_layout()
    plt.savefig("rq1_reachability.pdf", bbox_inches="tight")
    plt.show()

plot_reachability_grouped(PROJ_DF)

---
## 7. RQ1 — Total vs reachable functions (absolute numbers)

In [ ]:
def plot_total_vs_reached(proj_df: pd.DataFrame, case: str = "deep_no_stdlib"):
    sub      = proj_df[proj_df["case"] == case].copy()
    sub      = sub.sort_values("total_funcs", ascending=False)
    projects = sub["project"].tolist()

    x     = np.arange(len(projects))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(10, len(projects) * 1.6), 5))
    ax.bar(x - width/2, sub["total_funcs"],   width, label="Total functions",    color="steelblue", alpha=0.85, zorder=3)
    ax.bar(x + width/2, sub["reachable_funcs"], width, label="Reachable functions", color="seagreen",  alpha=0.85, zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(projects, rotation=30, ha="right")
    ax.set_ylabel("Function count")
    ax.set_title(f"RQ1 — Total vs reachable functions  [{CASE_LABELS[case]}]")
    ax.legend()
    ax.grid(axis="y", zorder=0)
    plt.tight_layout()
    plt.savefig(f"rq1_absolute_{case}.pdf", bbox_inches="tight")
    plt.show()

for c in CASE_ORDER:
    plot_total_vs_reached(PROJ_DF, c)

---
## 8. RQ2 — ACR distribution per project (box plot)

In [ ]:
def plot_acr_boxplot(pkg_df: pd.DataFrame, case: str = "deep_no_stdlib"):
    sub = pkg_df[
        (pkg_df["case"] == case) &
        (pkg_df["acr"].notna())
    ].copy()

    projects = sorted(sub["project"].unique())
    data     = [sub[sub["project"] == p]["acr"].values for p in projects]

    fig, ax = plt.subplots(figsize=(max(10, len(projects) * 1.6), 5))
    bp = ax.boxplot(
        data, labels=projects, patch_artist=True,
        medianprops=dict(color="black", linewidth=2),
    )
    for patch in bp["boxes"]:
        patch.set_facecolor("steelblue")
        patch.set_alpha(0.6)

    ax.set_xticklabels(projects, rotation=30, ha="right")
    ax.set_ylabel("API Coverage Ratio (ACR)")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title(f"RQ2 — ACR distribution across packages per project  [{CASE_LABELS[case]}]")
    ax.grid(axis="y", zorder=0)
    plt.tight_layout()
    plt.savefig(f"rq2_acr_boxplot_{case}.pdf", bbox_inches="tight")
    plt.show()

for c in CASE_ORDER:
    plot_acr_boxplot(PKG_DF, c)

---
## 9. RQ2 — Utilisation Index heatmap (top-N dependencies per project)

In [ ]:
def plot_ui_heatmap(ui_df: pd.DataFrame, case: str = "deep_no_stdlib",
                    top_n: int = 15, exclude_stdlib: bool = True):
    sub = ui_df[(ui_df["case"] == case)].copy()
    if exclude_stdlib:
        sub = sub[~sub["is_stdlib"]]

    # Keep top-N deps by mean UI across projects
    mean_ui  = sub.groupby("dep_group")["ui"].mean().nlargest(top_n).index
    sub      = sub[sub["dep_group"].isin(mean_ui)]

    pivot = sub.pivot_table(
        index="dep_group", columns="project", values="ui", aggfunc="sum"
    ).fillna(0)

    fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 1.5),
                                    max(6, len(pivot) * 0.5)))
    sns.heatmap(
        pivot, ax=ax, annot=True, fmt=".2f",
        cmap="YlOrRd", linewidths=0.5,
        cbar_kws={"label": "Utilisation Index (UI)"},
    )
    ax.set_title(f"RQ2 — Utilisation Index heatmap  [{CASE_LABELS[case]}]\n"
                 f"(top {top_n} third-party dependencies by mean UI)")
    ax.set_xlabel("Project")
    ax.set_ylabel("Dependency")
    plt.tight_layout()
    plt.savefig(f"rq2_ui_heatmap_{case}.pdf", bbox_inches="tight")
    plt.show()

for c in CASE_ORDER:
    plot_ui_heatmap(UI_DF, c)

---
## 10. RQ3 — Software Composition Index per project × configuration

In [ ]:
def plot_sci(proj_df: pd.DataFrame):
    cases    = CASE_ORDER
    projects = sorted(proj_df["project"].unique())
    n_proj   = len(projects)
    n_cases  = len(cases)

    x      = np.arange(n_proj)
    width  = 0.8 / n_cases
    offset = np.linspace(-(n_cases-1)/2, (n_cases-1)/2, n_cases) * width

    fig, ax = plt.subplots(figsize=(max(10, n_proj * 1.6), 5))

    for i, case in enumerate(cases):
        sub  = proj_df[proj_df["case"] == case].set_index("project")
        vals = [sub.loc[p, "sci"] if p in sub.index else np.nan for p in projects]
        ax.bar(
            x + offset[i], vals, width,
            label = CASE_LABELS[case],
            color = CASE_COLORS[case],
            alpha = 0.88,
            zorder=3,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(projects, rotation=30, ha="right")
    ax.set_ylabel("Software Composition Index (SCI)")
    ax.set_title("RQ3 — Fraction of reachable functions from third-party code (SCI)")
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(loc="upper right", framealpha=0.9)
    ax.grid(axis="y", zorder=0)
    plt.tight_layout()
    plt.savefig("rq3_sci.pdf", bbox_inches="tight")
    plt.show()

plot_sci(PROJ_DF)

---
## 11. RQ3 — Bloat: unused function share per dependency group

In [ ]:
def plot_bloat_heatmap(ui_df: pd.DataFrame, case: str = "deep_no_stdlib",
                       top_n: int = 15, exclude_stdlib: bool = True):
    """
    Heatmap of (1 - ACR) per dependency group × project.
    High values = high bloat.
    """
    sub = ui_df[(ui_df["case"] == case)].copy()
    if exclude_stdlib:
        sub = sub[~sub["is_stdlib"]]

    sub["bloat"] = 1 - sub["acr"].clip(0, 1)

    # Top-N by mean bloat
    mean_bloat = sub.groupby("dep_group")["bloat"].mean().nlargest(top_n).index
    sub        = sub[sub["dep_group"].isin(mean_bloat)]

    pivot = sub.pivot_table(
        index="dep_group", columns="project", values="bloat", aggfunc="mean"
    ).fillna(0)

    fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 1.5),
                                    max(6, len(pivot) * 0.5)))
    sns.heatmap(
        pivot, ax=ax, annot=True, fmt=".2f",
        cmap="RdYlGn_r", linewidths=0.5, vmin=0, vmax=1,
        cbar_kws={"label": "Bloat ratio  (1 – ACR)"},
    )
    ax.set_title(f"RQ3 — Dependency bloat heatmap  [{CASE_LABELS[case]}]\n"
                 f"(top {top_n} most bloated third-party dependencies)")
    ax.set_xlabel("Project")
    ax.set_ylabel("Dependency")
    plt.tight_layout()
    plt.savefig(f"rq3_bloat_heatmap_{case}.pdf", bbox_inches="tight")
    plt.show()

for c in CASE_ORDER:
    plot_bloat_heatmap(UI_DF, c)

---
## 12. RQ4 — Call site classification per project

In [ ]:
def plot_call_sites(proj_df: pd.DataFrame, case: str = "deep_no_stdlib"):
    sub      = proj_df[proj_df["case"] == case].copy()
    projects = sub.sort_values("total_sites", ascending=False)["project"].tolist()

    static_vals = []
    iface_vals  = []
    fvar_vals   = []

    for p in projects:
        row   = sub[sub["project"] == p].iloc[0]
        total = row["total_sites"] or 1
        static_vals.append(row["static_sites"]   / total)
        iface_vals.append( row["interface_sites"] / total)
        fvar_vals.append(  row["funcvar_sites"]   / total)

    x     = np.arange(len(projects))
    width = 0.55

    fig, ax = plt.subplots(figsize=(max(10, len(projects) * 1.6), 5))
    b1 = ax.bar(x, static_vals, width, label="Static",     color="steelblue", alpha=0.88, zorder=3)
    b2 = ax.bar(x, iface_vals,  width, label="Interface",  color="darkorange", alpha=0.88,
                bottom=static_vals, zorder=3)
    b3 = ax.bar(x, fvar_vals,   width, label="Func-var",   color="seagreen",   alpha=0.88,
                bottom=[s+i for s,i in zip(static_vals, iface_vals)], zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(projects, rotation=30, ha="right")
    ax.set_ylabel("Fraction of call sites")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title(f"RQ4 — Call-site classification  [{CASE_LABELS[case]}]")
    ax.legend(loc="lower right", framealpha=0.9)
    ax.grid(axis="y", zorder=0)
    plt.tight_layout()
    plt.savefig(f"rq4_callsites_{case}.pdf", bbox_inches="tight")
    plt.show()

for c in CASE_ORDER:
    plot_call_sites(PROJ_DF, c)

---
## 13. RQ4 — Signature-level target/call-site ratio

In [ ]:
def build_sig_df(raw: dict, case: str = "deep_no_stdlib") -> pd.DataFrame:
    rows = []
    for project, cases in raw.items():
        report = cases.get(case)
        if report is None:
            continue
        for sig, m in report.get("indirect", {}).get("signatureMetrics", {}).items():
            rows.append(dict(
                project          = project,
                signature        = sig,
                potential_targets= m.get("potentialTargets", 0),
                actual_sites     = m.get("actualCallSites",  0),
            ))
    return pd.DataFrame(rows)


def plot_sig_scatter(raw: dict, case: str = "deep_no_stdlib"):
    sig_df = build_sig_df(raw, case)
    if sig_df.empty:
        print("No signature data found.")
        return

    # Aggregate across projects for the ecosystem view
    agg = (
        sig_df
        .groupby("signature", as_index=False)
        .agg(potential_targets=("potential_targets","sum"),
             actual_sites      =("actual_sites",      "sum"))
    )
    # Only plot sigs with at least one actual call site
    agg = agg[agg["actual_sites"] > 0]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        agg["actual_sites"], agg["potential_targets"],
        alpha=0.65, edgecolors="k", linewidths=0.4, s=60, color="steelblue",
        zorder=3,
    )
    ax.set_xlabel("Actual indirect call sites")
    ax.set_ylabel("Potential target functions")
    ax.set_title(f"RQ4 — Signature ambiguity: targets vs call sites  [{CASE_LABELS[case]}]")
    ax.grid(zorder=0)
    plt.tight_layout()
    plt.savefig(f"rq4_sig_scatter_{case}.pdf", bbox_inches="tight")
    plt.show()

for c in CASE_ORDER:
    plot_sig_scatter(RAW, c)

---
## 14. Edge kind distribution per project  (supplementary)

In [ ]:
def build_edge_df(raw: dict) -> pd.DataFrame:
    rows = []
    for project, cases in raw.items():
        for case, report in cases.items():
            gt = report.get("grandTotal", {}).get("counts", {})
            for kind, count in gt.items():
                rows.append(dict(project=project, case=case,
                                 kind=kind, count=count))
    return pd.DataFrame(rows)


def plot_edge_distribution(raw: dict, case: str = "deep_no_stdlib"):
    edge_df  = build_edge_df(raw)
    sub      = edge_df[edge_df["case"] == case]
    if sub.empty:
        return

    pivot = sub.pivot_table(
        index="project", columns="kind", values="count", aggfunc="sum"
    ).fillna(0)

    # Normalise rows to fractions
    pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

    fig, ax = plt.subplots(figsize=(max(10, len(pivot) * 1.6), 5))
    pivot_norm.plot(kind="bar", stacked=True, ax=ax, alpha=0.88, zorder=3)

    ax.set_xlabel("Project")
    ax.set_ylabel("Fraction of edges")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title(f"Edge kind distribution  [{CASE_LABELS[case]}]")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    ax.legend(title="Edge kind", bbox_to_anchor=(1.01, 1), loc="upper left")
    ax.grid(axis="y", zorder=0)
    plt.tight_layout()
    plt.savefig(f"edges_{case}.pdf", bbox_inches="tight")
    plt.show()

for c in CASE_ORDER:
    plot_edge_distribution(RAW, c)

---
## 15. Summary statistics table

In [ ]:
def summary_table(proj_df: pd.DataFrame, pkg_df: pd.DataFrame):
    rows = []
    for case in CASE_ORDER:
        p  = proj_df[proj_df["case"] == case]
        pk = pkg_df[(pkg_df["case"] == case) & pkg_df["acr"].notna()]
        rows.append({
            "Configuration"           : CASE_LABELS[case],
            "Projects"                : len(p),
            "Median reachability"     : f"{p['reachability_ratio'].median():.1%}",
            "Median SCI"              : f"{p['sci'].median():.1%}",
            "Median ACR (pkg)"        : f"{pk['acr'].median():.1%}",
            "Median indirect ratio"   : f"{p['indirect_ratio'].median():.1%}",
        })
    return pd.DataFrame(rows).set_index("Configuration")

summary_table(PROJ_DF, PKG_DF)